In [6]:
import concurrent.futures
import geopandas as gpd
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import requests
from shapely.geometry import shape,Polygon,MultiPoint
import numpy as np
import datetime
import urllib3

urllib3.disable_warnings()

def calculate_convex_hull(coords):
    if not coords.size > 0 or len(coords) <= 3:  # Se la lista è vuota o meno di 3 punti
          
        return Polygon([])
    #else:
     #Crea un poligono dalle coordinate
    multi_point = MultiPoint(coords)
    return multi_point.convex_hull

def get_max_record_count(fs_url):
    response = requests.get(url = fs_url, params = {'f':'json'}, verify = False)
    layer_info = response.json()
    max_record_count = layer_info['maxRecordCount']
    return max_record_count

def fetch_batch(URL, offset, count):
    res = requests.get(
        url=f"{URL}/query",
        params={
            'where': '1=1',
            'outFields': '*',
            'resultOffset': offset,
            'resultRecordCount': count,
            'f': 'pgeojson'
        },
        verify=False
    )
    res_json = res.json()
    features = res_json['features']
    geometries = [shape(f['geometry']) for f in features]
    attributes = [f['properties'] for f in features]
    df = pd.DataFrame(attributes)
    df['geometry'] = geometries
    return df



def get_esri_service_parallel(service):
    URL = f"https://services9.arcgis.com/RHVPKKiFTONKtxq3/ArcGIS/rest/services/{service}/FeatureServer/0"
    # Ottieni il count totale
    response = requests.get(
        url=f"{URL}/query",
        params={
            'where': '1=1',
            'returnCountOnly': True,
            'f': 'pjson'
        },
        verify=False
    )
    total_record_count = response.json()['count']
    max_record_count = get_max_record_count(URL)  # tua funzione esistente
    # Organizza gli offset in modo da fare più richieste in parallelo

    offsets = []
    current_offset = 0
    while current_offset < total_record_count:
        count = min(max_record_count, total_record_count - current_offset)
        offsets.append((current_offset, count))
        current_offset += count

    # Parallelizza le richieste
    new_df_list = []
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = [executor.submit(fetch_batch, URL, off, cnt) for off, cnt in offsets]
        for future in concurrent.futures.as_completed(futures):
            df_part = future.result()
            new_df_list.append(df_part)
    # Combina tutto
    new_df = pd.concat(new_df_list, ignore_index=True)
    gdf = gpd.GeoDataFrame(new_df, geometry='geometry', crs='EPSG:4326')
    return gdf

# Call the ESRI service in parallel:

viirs_points = get_esri_service_parallel("Satellite_VIIRS_Thermal_Hotspots_and_Fire_Activity")
print("len viirs:",len(viirs_points))

modis_48h_points = get_esri_service_parallel("MODIS_Thermal_v1")
print("len modis",len(modis_48h_points))



modis_48h_points['latitude']= modis_48h_points['geometry'].y
modis_48h_points['longitude']= modis_48h_points['geometry'].x

df_points = gpd.GeoDataFrame(pd.concat([viirs_points, modis_48h_points], ignore_index=True))
'''
gdf_corine_industrial = gpd.read_file("corine_industrial_wgs84_buffer300.gpkg")
sjoin_result = gpd.sjoin(df_points, gdf_corine_industrial, how='inner', predicate='intersects')

# Ottieni gli indici dei punti che ricadono nelle aree industriali
indices_industrial = sjoin_result.index

# Filtra i punti che NON ricadono nelle aree industriali
df_points = df_points[~df_points.index.isin(indices_industrial)]
'''
print(len(df_points))

# Convertire le coordinate in un formato adatto per DBSCAN
coordinates = df_points[['longitude', 'latitude']].to_numpy()

# min_saples e eps
min_samples=4
#eps_selected=get_eps(min_samples,coordinates)
eps_selected=3 #in km

# Esegui il clustering con DBSCAN
db = DBSCAN(eps=eps_selected/6371., min_samples=min_samples, algorithm='ball_tree').fit(np.radians(coordinates))
cluster_labels = db.labels_
num_clusters = len(set(cluster_labels))
clusters = pd.Series([coordinates[cluster_labels == n] for n in range(num_clusters)])
print('Number of clusters: {}'.format(num_clusters))

# Applicare la funzione a ogni riga della Serie
convex_hulls = clusters.apply(calculate_convex_hull)

#polygons = convex_hulls.apply(lambda x: shapely.wkt.loads(x) if x != "POLYGON EMPTY" else None)
gdf_polygons = gpd.GeoDataFrame(geometry=convex_hulls)

# Imposta il CRS (Coordinate Reference System) se necessario
gdf_polygons.set_crs(epsg=4326, inplace=True) 

gdf_polygons = gdf_polygons[gdf_polygons.geometry.apply(lambda x: not x.is_empty)]



len viirs: 1576251
len modis 28895
1605146
Number of clusters: 38015


In [5]:
df_points.to_file(f"./points_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.geojson", driver='GeoJSON')
gdf_polygons.to_file(f"./wildcap_polygons_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.geojson", driver='GeoJSON')

In [7]:
#clip df_points to spatial extent

spatial_extent = {"type":"Polygon","coordinates":[[[14.347836,40.772471],[14.347836,40.852833],[14.490726,40.852833],[14.490726,40.772471],[14.347836,40.772471]]]}
df_points_clipped = df_points[df_points.geometry.within(shape(spatial_extent))]

df_points_clipped.to_file(f"./points_clipped_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.geojson", driver='GeoJSON')
gdf_polygons_clipped = gdf_polygons[gdf_polygons.geometry.within(shape(spatial_extent))]
gdf_polygons_clipped.to_file(f"./wildcap_polygons_clipped_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.geojson", driver='GeoJSON')

In [12]:
gdf_polygons_clipped.set_crs(epsg=4326, inplace=True)
gdf_polygons_clipped.to_crs(epsg=32633, inplace=True)  # Converti in UTM zone 33N

/usr/local/python/3.12.1/lib/python3.12/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [ ]:
gdf_polygons_clipped.geometry.area


26255    9.680663e+06
dtype: float64